In [ ]:
import json
import pandas as pd
import rasterio
import os
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import convolve2d

In [ ]:
rundir = "/home/ebr/projects/release-volume-sampler/generated/messina_001"
volumes_path = os.path.join(rundir,"volumes/recursive_propagations.json")
# Load the JSON file
with open(volumes_path, "r") as f:
    data = json.load(f)

triangulation_dir = os.path.join(rundir, "triangulation")
tri_mask_path = os.path.join(triangulation_dir, "triangulation.tif")

In [ ]:
# Normalize the "volumes" data within each entry
df = pd.json_normalize(
    data,
    record_path=["volumes"],  # Extract and flatten the "volumes" list
    meta=["seed_triangle", "seed_triangle_probability"]  # Include these fields as metadata
).astype({"seed_triangle": int, "seed_triangle_probability": float})

# Keep 'released' as a list and skip expanding 'steps'
# Drop 'steps' if it's not needed
df = df.drop(columns=["steps"])
df['id'] = df.index

In [ ]:
df["probability"] = df.seed_triangle_probability*df.condprob # Use seed probability to set final probability
df["probability"] = df["probability"]/df["probability"].sum()  # Normalize

In [ ]:
df.dtypes

In [ ]:
df.shape

Statistical relation for assigning depth of the volume (Zengafinnen-Morris et al. 2022): $V = 0.0298*A^{1.36}$


In [ ]:
df["volume"] = 0.0298*df.area**1.36
df["thickness"] = df.volume/df.area

In [ ]:
df.area.plot(kind="hist", weights=df.probability, bins=30, title="Weighted area of release")

In [ ]:
df.thickness.plot(kind="hist", weights=df.probability, bins=30, title="Weighted thickness of release")

## Make density plot of release.

In [ ]:
df.released

In [ ]:
with rasterio.open(tri_mask_path) as src:
    tri_mask = src.read(1)  # Read the triangle mask
    tri_profile = src.profile  # Copy metadata to use in output

In [ ]:
from matplotlib.colors import LogNorm

n_triangles = int(tri_mask.max()) + 1
counts = np.zeros(n_triangles)
probs = np.zeros(n_triangles)

for i, row in df.iterrows():
    counts[row.released] += 1.
    probs[row.released] += row.probability

counts_raster = np.zeros(tri_mask.shape)
prob_raster = np.zeros(tri_mask.shape)

for tri_index in range(n_triangles):
    counts_raster += np.where(tri_mask == tri_index, counts[tri_index], 0)
    prob_raster += np.where(tri_mask == tri_index, probs[tri_index], 0)

In [ ]:

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 1 row, 2 columns of plots
fig.suptitle("Spatial distribution of release volumes", fontsize=16)

# Plot the first raster
counts_im = axes[0].imshow(
    counts_raster,
    cmap="tab20b",
    norm=LogNorm(vmin=1, vmax=np.nanmax(counts_raster), clip=True),
)

# Plot the second raster
prob_im = axes[1].imshow(
    prob_raster,
    cmap="tab20b",
    norm=LogNorm(vmin=1e-6, vmax=np.nanmax(prob_raster), clip=True),
)

# Create colorbars with the same height as the subplots
from mpl_toolkits.axes_grid1 import make_axes_locatable

# For the first axis
divider1 = make_axes_locatable(axes[0])
cax1 = divider1.append_axes("right", size="5%", pad=0.1)  # Adjust size and padding
fig.colorbar(counts_im, cax=cax1, label="Counts")

# For the second axis
divider2 = make_axes_locatable(axes[1])
cax2 = divider2.append_axes("right", size="5%", pad=0.1)  # Adjust size and padding
fig.colorbar(prob_im, cax=cax2, label="Probability")

# Adjust layout
plt.tight_layout(rect=[0, 0, 1, 0.95])  # Leave space for the title
plt.show()


In [ ]:
# write volumes to rasters
df_filtered = df.loc[(df.mean_elevation > -500) & (df.area > 1300000) & (df.probability > 1e-6)]
df_filtered.sort_values("area", ascending=False)

In [ ]:
df["tsunami_potential_ratio"] = df.volume/(0.0957*(df.mean_slope**-1.609)*((0-df.mean_elevation)**1.3807)*1e6)

In [ ]:
df.tsunami_potential_ratio.hist(bins=30, range=[0,10])

In [ ]:
df_filtered = df[df.tsunami_potential_ratio > 1.]

In [ ]:
df_filtered = df_filtered.sort_values("tsunami_potential_ratio", ascending=False)

In [ ]:
df_filtered.iloc[:10].shape

In [ ]:
df.tsunami_potential_ratio > 2.

In [ ]:
with rasterio.open(tri_mask_path) as src:
    tri_mask = src.read(1)  # Read the triangle mask
    tri_profile = src.profile  # Copy metadata to use in output

with rasterio.open("/home/ebr/projects/release-volume-sampler/generated/messina_001/bathy_truncated.tif") as src:
    bathy = src.read(1)  # Read the triangle mask
    bathy_profile = src.profile  # Copy metadata to use in output

# Update profile for single-band, unsigned 8-bit data
bathy_profile.update(dtype=rasterio.float32, count=1)

#added_volumes = np.zeros((profile["height"], profile["width"]))

for i, volume in df_filtered.iterrows():
    print(f"i: {i}, volume:{volume}")
    
    # Create binary volume mask: 1 if pixel belongs to specified triangles, else 0
    volume_mask = np.isin(tri_mask, volume.released)
    

    # Smooth volume
    volume_raster = convolve2d(volume_mask.astype(float)*volume.thickness, np.ones((3,3))/9., mode="same")
    
    volume_path = os.path.join(rundir, f"volumes/rasters/volume_seed-{volume.seed_triangle}_area-{volume.area:.2e}_id-{volume.id}.tif")

    with rasterio.open(volume_path, 'w', **bathy_profile) as dst:
        dst.write(volume_raster.astype(rasterio.float32), 1)  # Write volume to file

In [ ]:
bathy_profile

In [ ]:

with rasterio.open(tri_mask_path) as src:
    tri_mask = src.read(1)  # Read the triangle mask
    tri_profile = src.profile  # Copy metadata to use in output

with rasterio.open("/home/ebr/projects/release-volume-sampler/generated/messina_001/bathy_truncated.tif") as src:
    bathy = src.read(1)  # Read the triangle mask
    bathy_profile = src.profile  # Copy metadata to use in output

# Update profile for single-band, unsigned 8-bit data
bathy_profile.update(dtype=rasterio.float32, count=1)
bathy_profile.update(driver="AAIGrid")

#added_volumes = np.zeros((profile["height"], profile["width"]))

for i, volume in df_filtered.iterrows():
    print(f"i: {i}, volume:{volume}")
    
    # Create binary volume mask: 1 if pixel belongs to specified triangles, else 0
    volume_mask = np.isin(tri_mask, volume.released)
    

    # Smooth volume
    volume_raster = convolve2d(volume_mask.astype(float)*volume.thickness, np.ones((3,3))/9., mode="same")
    
    volume_path = os.path.join(rundir, f"volumes/rasters/volume_seed-{volume.seed_triangle}_area-{volume.area:.2e}_id-{volume.id}.asc")

    with rasterio.open(volume_path, 'w', **bathy_profile) as dst:
        dst.write(volume_raster.astype(rasterio.float32), 1)  # Write volume to file
    break

In [ ]:
bathy_profile.update(driver="aaigrid")

In [ ]:
bathy_profile

In [ ]:
bathy_profile

In [ ]:
# Create a figure and axis objects
fig, axes = plt.subplots(2, 2, figsize=(12, 8))  # 2 rows, 2 columns of plots
fig.suptitle("Release Characteristics", fontsize=16)

# Top-left: Histogram of area
axes[0, 0].hist(df.area, bins=30, color="skyblue", edgecolor="black")
axes[0, 0].set_title("Area of Release")
axes[0, 0].set_xlabel("Area")
axes[0, 0].set_ylabel("Frequency")

# Top-right: Histogram of thickness
axes[0, 1].hist(df.thickness, bins=30, color="lightgreen", edgecolor="black")
axes[0, 1].set_title("Thickness of Release")
axes[0, 1].set_xlabel("Thickness")
axes[0, 1].set_ylabel("Frequency")

# Bottom-left: Weighted histogram of area
axes[1, 0].hist(df.area, bins=30, weights=df.probability, color="salmon", edgecolor="black")
axes[1, 0].set_title("Weighted Area of Release")
axes[1, 0].set_xlabel("Area")
axes[1, 0].set_ylabel("Weighted Frequency")

# Bottom-right: Weighted histogram of thickness
axes[1, 1].hist(df.thickness, bins=30, weights=df.probability, color="orange", edgecolor="black")
axes[1, 1].set_title("Weighted Thickness of Release")
axes[1, 1].set_xlabel("Thickness")
axes[1, 1].set_ylabel("Weighted Frequency")

# Adjust layout to avoid overlap
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Leave space for the main title

# Save or show the plot
#plt.savefig("release_characteristics.png")  # Save as an image file
plt.show()  # Display the figure

## Find slopeunits

In [ ]:
import os
import numpy as np


In [ ]:
os.chdir("/home/ebr/projects/release-volume-sampler")

In [ ]:
from src.volume_sampler.release_volume_sampler import RecursiveReleaseAnalysis

# Usage example
config = {
    "rundir": "/home/ebr/projects/release-volume-sampler/generated/messina_001",
    "mesh_path": "/home/ebr/projects/release-volume-sampler/generated/messina_001/triangulation/triangulation.vtk",
    "cumprob_logfos_path": "/home/ebr/projects/release-volume-sampler/generated/messina_001/triangulation/cummulative_fos.npz",
    "utm_epsg_code": 32633, # Messina strait
}

run_config = {
    "fos_threshold": 1.1,
    "recursive_probability_threshold": 0.01,
    "seed_triangle_probability_threshold": 0.1,
}
# Execute analysis.
analysis = RecursiveReleaseAnalysis(**config)

In [ ]:
r2 = (analysis.normals**2).sum(axis=1)
slope = np.rad2deg(np.arccos(1/np.sqrt(r2)))

In [ ]:
plt.hist(analysis.slopes, bins=40)

In [ ]:
analysis.slopes[51]

In [ ]:
slopeunits = np.zeros(analysis.n_triangles, dtype=int)
triangles = list(range(analysis.n_triangles))
slopeunit = 1
while(triangles):
    print(len(triangles))
    triangle = triangles.pop()
    next_upstream = analysis.get_upstream_triangles(triangle).tolist()
    
    upstream_triangles = []
    while(next_upstream):
        previous_upstream = next_upstream.copy()
        next_upstream = []
        for t in previous_upstream:
            next_upstream.extend(analysis.get_upstream_triangles(t).tolist())
        upstream_triangles.extend(next_upstream)
    
    # Remove duplicates
    upstream_triangles = list(set(upstream_triangles))
    if len(upstream_triangles) > 0:
        upstream_slopeunits = slopeunits[np.array(upstream_triangles)]
        if upstream_slopeunits.max() == 0:
            slopeunits[triangle] = slopeunit
            slopeunits[upstream_triangles] = slopeunit
            [triangles.remove(t) for t in upstream_triangles]
        else:
            slopeunits[triangle] = upstream_slopeunits.max()
    else:
        slopeunit += 1
        slopeunits[triangle] = slopeunit
    
    #print(f"triangle: {triangle}, upstream: {upstream_slopeunits.max()}")

## Load shakemap data and estimate probabilities.

In [ ]:
import os
import numpy as np
from scipy.interpolate import interp1d

os.chdir("/home/ebr/projects/release-volume-sampler")

In [ ]:
from src.volume_sampler.volume_writer import VolumeWriter

In [ ]:

# Usage example
config = {
    "rundir": "/home/ebr/projects/release-volume-sampler/generated/messina_001",
}

filter_config = {
    "tsunami_potential_ratio_threshold": 1.,
    "max_rasters": 1000,
}

# Execute
writer = VolumeWriter(**config)
#writer.write_volumes_to_csv()
#writer.plot_distribution()
#writer.plot_release_density_plots()

In [ ]:
writer

In [ ]:

# Load lookuptable
lookup_table = os.path.join(writer.triangulation_dir, "exceedance_displacement.npz")
diplacement_exceedance = np.load(lookup_table)
thresholds, exceedance_probs = diplacement_exceedance["thresholds"], diplacement_exceedance["probs"]

threshold = 5.
n_triangles = writer.tri_mask.max()+1


In [ ]:
interpolator = interp1d(x=thresholds, y=exceedance_probs, fill_value=(1.,0.), bounds_error=False) # assumes that the entire range is essentially covered.

In [ ]:
writer.df["seed_triangle_probability_shake"] = interpolator(threshold)[writer.df.seed_triangle.to_numpy()]

In [ ]:
writer.df.head()

In [ ]:
writer.df["seed_triangle_probability_shake"].hist(bins=30)

In [ ]:
writer.df["seed_triangle_probability"].hist(bins=30)

In [ ]:
np.arange(4, dtype=int)